In [1]:
# A set of functions to do analytical transformations of equations

In [2]:
mutable struct Term
    spin_types::String
    spin_indices::String
    bosons::String
    exponents::Vector{Int}
    coeff::ComplexF64
    vars::Vector{String}
    conjugate::Bool

    # Inner constructor with default values
    function Term(; spin_types::String="", spin_indices::String="", bosons::String="", exponents::Vector{Int}=Int[], coeff::ComplexF64=0.0 + 0.0im, vars::Vector{String}=String[], conjugate::Bool=false)
        new(spin_types, spin_indices, bosons, exponents, coeff, vars, conjugate)
    end
end
# Test
term = Term(spin_types="xy", spin_indices="ij")
#term.spin_types = "xy"
#term.spin_indices = "ij"
term.bosons = "+-"
term.exponents = [1, 2]
term.coeff = 1.0+0.0im
print(term)

Term("xy", "ij", "+-"

, [1, 2], 1.0 + 0.0im, String

[])

In [3]:
function str2int_array(s::String)::Vector{Int}
    # transform string into integer array with an integer for each character
    return Int.(collect(s))
end

function int_array2str(a::Vector{Int})::String
    # transform integer array into string
    return join(Char.(a))
end

function sort_spins_by_index!(term::Term)
    indices::String = term.spin_indices
    types::String = term.spin_types
    int_indices = str2int_array(indices)
    ind = sortperm(int_indices)
    int_indices = int_indices[ind]
    term.spin_types = types[ind]
    term.spin_indices = int_array2str(int_indices)
end
# test
term = Term(spin_types="xyz", spin_indices="kji")
display(term)
sort_spins_by_index!(term)
display(term)

Term("xyz", "kji", "", Int[], 0.0 + 0.0im, String[])

Term("zyx", "ijk", "", Int[], 0.0 + 0.0im, String[])

In [4]:
function type_chars2index(s::String)::Vector{Int}
    # transforms xyz+- into 12345
    indexes = str2int_array(s)
    ind_list = indexes .- 119
    ind_list[ind_list .== -76] .= 4
    ind_list[ind_list .== -74] .= 5
    # if any ind_list not 1 to 5 then error
    if any(ind_list .< 1) || any(ind_list .> 5)
        error("type_chars2index: invalid character in string")
    end
    return ind_list
end

function index2type_chars(indexes::Vector{Int})::String
    # transforms 12345 into xyz+-
    type_chars = ['x', 'y', 'z', '+', '-']
    str = ""
    for i in indexes
        if i < 1 || i > 5
            error("index2type_chars: invalid index")
        end
        str *= type_chars[i]
    end
    return str
end

# test
indexes = type_chars2index("xyz+-")
display(indexes)
string = index2type_chars(indexes)
display(string)

5-element Vector{Int}:
 1
 2
 3
 4
 5

"xyz+-"

In [5]:
function get_spin_coeff()::Tuple{Matrix{Int},Matrix{ComplexF64}}
    # Generates a cayley table for the spin product coefficients, separated into operator index and coefficient
    # Returns tables for spin product coefficients x,y,z,I are on indexes i= 1,2,3,4 respectively
    ij_k = [4 3 2 1; 3 4 1 2; 2 1 4 3; 1 2 3 4]
    ij_coeff = ComplexF64[1 im -im 1; -im 1 im 1; im -im 1 1; 1 1 1 1]
    return ij_k, ij_coeff
end

# test
ij_k, ij_coeff = get_spin_coeff()
display(ij_k)
display(ij_coeff)

4×4 Matrix{Int}:
 4  3  2  1
 3  4  1  2
 2  1  4  3
 1  2  3  4

4×4 Matrix{ComplexF64}:
 1.0+0.0im  0.0+1.0im  0.0-1.0im  1.0+0.0im
 0.0-1.0im  1.0+0.0im  0.0+1.0im  1.0+0.0im
 0.0+1.0im  0.0-1.0im  1.0+0.0im  1.0+0.0im
 1.0+0.0im  1.0+0.0im  1.0+0.0im  1.0+0.0im

In [6]:
function simplify_spin_terms!(term::Term)
    ij_k, ij_coeff = get_spin_coeff()
    spin_types = term.spin_types
    spin_indices = term.spin_indices
    coeff = term.coeff
    n = length(spin_types)
    if length(spin_types) != length(spin_indices)
        throw(ArgumentError("Invalid spin term, spin_types and spin_indices must be the same length"))
    end
    if n >= 2
        spin_type_ind = type_chars2index(spin_types)
        i = 1
        while i <= n - 1
            if spin_indices[i] == spin_indices[i + 1]
                j = spin_type_ind[i]
                k = spin_type_ind[i + 1] 
                new_type = ij_k[j, k] 
                new_coeff = ij_coeff[j, k]
                coeff *= new_coeff

                if new_type == 4
                    deleteat!(spin_type_ind, [i, i + 1])
                    spin_indices = spin_indices[1:i-1] * spin_indices[i+2:end]
                    n -= 2
                else
                    spin_type_ind[i] = new_type 
                    deleteat!(spin_type_ind, i + 1)
                    spin_indices = spin_indices[1:i] * spin_indices[i+2:end]
                    n -= 1
                end
            else
                i += 1
            end
        end
        spin_types = join(index2type_chars(spin_type_ind))
    end
    term.spin_types = spin_types
    term.spin_indices = spin_indices
    term.coeff = coeff
end

# test
spin_types = "xyzyyzzx"
spin_indices = "iikiijjj"
term = Term(spin_types=spin_types, spin_indices=spin_indices)
term.coeff = 1.0+0.0im
display(term)
simplify_spin_terms!(term)
display(term)

Term("xyzyyzzx", "iikiijjj", "", Int[], 1.0 + 0.0im, String[])

Term("zzx", "ikj", "", Int[], 0.0 + 1.0im, String[])

In [7]:
function clean_up_bosons!(term::Term)
    # Check for double -- and ++, reduce them to - and +, respectively and add up their exponents
    bosons = term.bosons
    exponents = term.exponents
    i = 1
    if length(bosons) != length(exponents)
        throw(ArgumentError("Invalid boson types and exponents, not same length"))
    end

    while i < length(bosons)
        if bosons[i] == bosons[i+1]
            # Same type, add exponents and remove one
            exponents[i] += exponents[i+1]
            bosons = bosons[1:i] * bosons[i+2:end]
            deleteat!(exponents, i+1)
        else
            i += 1
        end
    end
    term.bosons = bosons
    term.exponents = exponents
end


# test
term.bosons = "++--"
term.exponents = [1, 1, 1, 1]
display(term)
clean_up_bosons!(term)
display(term)

Term("zzx", "ikj", "++--", [1, 1, 1, 1], 0.0 + 1.0im, String[])

Term("zzx", "ikj", "+-", [2, 2], 0.0 + 1.0im, String[])

In [8]:
function remove_borders!(term::Term)
    bosons = term.bosons
    exponents = term.exponents
    bosons, exponents = clean_up_bosons(bosons, exponents)
    if !isempty(bosons)
        if exponents[1] == 0
            bosons = bosons[2:end]
            exponents = exponents[2:end]
        end
        if exponents[end] == 0
            bosons = bosons[1:end-1]
            exponents = exponents[1:end-1]
        end
    end
    term.bosons = bosons
    term.exponents = exponents
end

function add_empty_borders!(term::Term)
    bosons = term.bosons
    exponents = term.exponents
    if !isempty(bosons)
        if bosons[1] != '+'
            bosons = "+" * bosons
            exponents = vcat(0, exponents)
        end
        if bosons[end] != '-'
            bosons = bosons * "-"
            exponents = vcat(exponents, 0)
        end
    else
        bosons = "+-"
        exponents = [0, 0]
    end
    term.bosons = bosons
    term.exponents = exponents
end

function remove_empty_elements!(term::Term, borders::Bool=false)
    # Remove empty elements from bosons and exponents and add empty borders if needed
    clean_up_bosons!(term)
    bosons = term.bosons
    exponents = term.exponents
    border_int::Int = borders ? 0 : 1
    i = border_int + 1

    while i <= length(exponents) - border_int
        if exponents[i] == 0
            bosons = bosons[1:i-1] * bosons[i+1:end]
            exponents = vcat(exponents[1:i-1], exponents[i+1:end])
        else
            i += 1
        end
    end
    term.bosons = bosons
    term.exponents = exponents
    clean_up_bosons!(term)
end

# test
term.bosons = "++++-"
term.exponents = [0, 1, 0, 1, 0]
display(term)
remove_empty_elements!(term)
display(term)
display("-"^50)
term.bosons = "+-"
term.exponents = [0, 0]
display(term)
remove_empty_elements!(term, true)
display(term)

Term("zzx", "ikj", "++++-", [0, 1, 0, 1, 0], 0.0 + 1.0im, String[])

Term("zzx", "ikj", "+-", [2, 0], 0.0 + 1.0im, String[])

"--------------------------------------------------"

Term("zzx", "ikj", "+-", [0, 0], 0.0 + 1.0im, String[])

Term("zzx", "ikj", "", Int[], 0.0 + 1.0im, String[])

In [11]:
function make_term(str_term::String, coeff=1.0+0im, vars::Vector{String}=String[], conjugate::Bool=false)::Term
    if !isa(coeff, ComplexF64)
        coeff = ComplexF64(coeff)
    end

    # Create Term struct from string
    term = Term()
    indices::String = ""
    types::String = ""

    op_types = ['x', 'y', 'z', '+', '-']
    str_term = replace(str_term, "_"=>"")
    for s in op_types
        str_term = replace(str_term, s=>"*" * s)
    end
    string_list = split(str_term, "*")
    # remove empty strings in op_list
    string_list = filter(p -> p != "", string_list)
    for (i, p) in enumerate(string_list)
        if p[1] in ['x', 'y', 'z']
            types *= p[1]
            if length(p) == 1
                indices *= " "
            else
                indices *= p[2:end]
            end
        elseif p[1] in ['+', '-']
            term.bosons *= p[1]
        else
            error("Operator not recognized")
        end
    end
    term.spin_types = types
    term.spin_indices = indices
    term.coeff = coeff
    term.vars = vars
    term.exponents = ones(Int, length(term.bosons))
    term.conjugate = conjugate
    ## Add post processing
    # 1st sort spins by sort_spins_by_index
    sort_spins_by_index!(term)
    # 2nd simplify spin terms
    simplify_spin_terms!(term)
    # 3rd clean up bosons
    clean_up_bosons!(term)
    return term
end


# test the function
term = make_term("+-x_j*yi")
display(term)
term = make_term("+-xjyi", 1.0+0.0im, ["a", "b"])
display(term)
term = make_term("xy", 1.0+0.0im, ["a", "b"])
display(term)

Term("yx", "ij", "+-", [1, 1], 1.0 + 0.0im, String[])

Term("yx", "ij", "+-", [1, 1], 1.0 + 0.0im, ["a", "b"])

Term("z", " ", "", Int[], 0.0 + 1.0im, ["a", "b"])

In [25]:
function terms_combinable(A::Term, B::Term)
    if !(A.spin_types == B.spin_types)
        return false
    elseif !(A.spin_indices == B.spin_indices)
        return false
    elseif !(A.bosons == B.bosons)
        return false
    elseif !(A.exponents == B.exponents)
        return false
    elseif !(A.vars == B.vars)
        return false
    else
        return true
    end
end

function combine_terms(terms::Vector{Term}, eps::Float64=1e-14)::Vector{Term}
    term_list = deepcopy(terms)
    i = 1
    while i <= length(term_list)
        j = i + 1
        while j <= length(term_list)
            if terms_combinable(term_list[i], term_list[j])
                term_list[i].coeff += term_list[j].coeff
                deleteat!(term_list, j)
            else
                j += 1
            end
        end
        i += 1
    end
    # remove vanishingly small terms
    term_list = filter(term -> abs(term.coeff) > eps, term_list)
    return term_list
end

# test
terms = [make_term("+y_j*x_i"), make_term("+x_i*y_j"), make_term("+x_i*y_j")]
term_list = combine_terms(terms)
display(term_list[1])

Term("xy", "ij", "+", [1], 3.0 + 0.0im, String[])

In [32]:
function normal_order(term::Term)::Vector{Term} 
    # Create normal ordering of a term, returns a Vector of Terms)
    function flip_first_plus_element!(term::Term)::Term
        remove_empty_elements!(term, false)
        term_copy = deepcopy(term)
        bosons = term.bosons
        exponents = term.exponents
        if length(term.bosons) > 2 && term.exponents[3] > 0
            if term.bosons[2] != '-'
                error("Invalid boson types, bosons[2] is not -. (", term.bosons, ", ", term.exponents, ")")
            end
            if term.bosons[3] != '+'
                error("Invalid boson types, bosons[3] is not +. (", term.bosons, ", ", term.exponents, ")")
            end
            multiplier = term.exponents[2]
            term_copy.exponents[2] -= 1
            term_copy.exponents[3] -= 1
            term.exponents[1] += 1
            term.exponents[3] -= 1
            term_copy.coeff *= multiplier
            
            remove_empty_elements!(term, false)
            remove_empty_elements!(term_copy, false)
        else
            term_copy.coeff = 0.0 + 0.0im
        end
        return term_copy
    end
    term2 = deepcopy(term)
    remove_empty_elements!(term2, true)
    add_empty_borders!(term2)
    
    term_copy = flip_first_plus_element!(term2)
    term_array::Vector{Term} = Term[]
    if term_copy.coeff == 0
        term_array = [term]
    else
        term_array = normal_order(term2)
        append!(term_array, normal_order(term_copy))
    end
    # remove duplicates and vanishingly small terms
    for i in 1:length(term_array)
        remove_empty_elements!(term_array[i], true)
    end
    term_array = combine_terms(term_array)
    return term_array
end
# Test  
term.bosons = "-+-+"
term.exponents = [0,2,2,1]
term.coeff = 1.0+0.0im
display(term)
term_array = normal_order(term)
for term in term_array
    bos = term.bosons
    expo = term.exponents
    coeff = term.coeff
    println(bos, " ", expo, " ", coeff)
end

Term("xy", "ij", "-+-+", [0, 2, 2, 1], 1.0 + 0.0im, String[])

+- [3, 2] 1.0 + 0.0im
+- [2, 1] 2.0 + 0.0im


In [17]:
function add_terms(termA::Union{Term, Vector{Term}}, termB::Union{Term, Vector{Term}})::Vector{Term}
    # Adds two terms together
    term_list::Vector{Term} = Term[]
    if isa(termA, Term)
        termA = Term[termA]
    end
    if isa(termB, Term)
        termB = Term[termB]
    end
    append!(term_list, termA)
    append!(term_list, termB)
    term_list = combine_terms(term_list)
    return term_list
end
# Test
termA = Term[make_term("+x_i*y_j")]
termB = make_term("+-x_i*y_j")
term_list = add_terms(termA, termB)
display(term_list)

2-element Vector{Term}:
 Term("xy", "ij", "+", [1], 1.0 + 0.0im, String[])
 Term("xy", "ij", "+-", [1, 1], 1.0 + 0.0im, String[])

In [13]:
function multiply_terms(A::Union{Term, Vector{Term}}, B::Union{Term, Vector{Term}}; normalize::Bool=true)::Union{Term, Vector{Term}}
    if isa(A, Vector{Term})
        res = []
        for term in A
            curr_res = multiply_terms(term, B, normalize=normalize)
            for c in curr_res
                if !isa(c, Vector{Term})
                    push!(res, c)
                else
                    append!(res, c)
                end
            end
        end
        return combine_terms(res)
    elseif isa(B, Vector{Term})
        res::Vector{Term} = []
        for term in B
            curr_res = multiply_terms(A, term, normalize=normalize)
            for c in curr_res
                if !isa(c, Vector{Term})
                    push!(res, c)
                else
                    append!(res, c)
                end
            end
        end
        return combine_terms(res)
    else
        # Add implementation for `get_spin_coeff`
        ij_k, ij_coeff = get_spin_coeff()

        # Make sure to define and import all used methods and fields here
        spin_types = A.spin_types * B.spin_types
        spin_indices = A.spin_indices * B.spin_indices
        bosons = A.bosons * B.bosons
        exponents = vcat(A.exponents, B.exponents)
        coeff = A.coeff * B.coeff
        vars = filter(x -> !isempty(x), vcat(A.vars, B.vars))
        term = Term(spin_types=spin_types, spin_indices=spin_indices, bosons=bosons, exponents=exponents, coeff=coeff, vars=vars)
        sort_spins_by_index!(term)
        simplify_spin_terms!(term)
    	clean_up_bosons!(term)
        if normalize
            return normal_order(term)
        end
        return term
    end
end

# Generate two terms that can be multiplied
A = make_term("+--x_i*y_j")
B = make_term("+y_i*x_j")
display(A)
display(B)
C = multiply_terms(A, B)
display(C)


Term("xy", "ij", "+--", [1, 1, 1], 1.0 + 0.0im, String[])

Term("yx", "ij", "+", [1], 1.0 + 0.0im, String[])

2-element Vector{Term}:
 Term("zz", "ij", "+-", [2, 2], 1.0 + 0.0im, String[])
 Term("zz", "ij", "+-", [1, 1], 2.0 + 0.0im, String[])

In [149]:
using LinearAlgebra: conj

function dagger_term(term::Union{Term, Vector{Term}}; normalize::Bool=true, eps::Float64=1e-14)::Union{Term, Vector{Term}}
    # Does not conjugate vars!
    if isa(term, Vector)
        res::Vector{Term} = []
        for t in term
            curr_res = dagger_term(t, normalize=normalize, eps=eps)
            if !isa(curr_res, Vector{Term})
                push!(res, curr_res)
            else
                append!(res, curr_res)
            end
        end
        return res
    else
        dag_term = deepcopy(term) 
        # Change the signs of bosons and reverse them 
        dag_term.bosons = reverse(term.bosons)
        # replace + with - and - with +
        dag_term.bosons = map(x -> x == '+' ? '-' : '+', dag_term.bosons)
        dag_term.exponents = reverse(dag_term.exponents) # reverse exponents order
        dag_term.coeff = conj(dag_term.coeff) # conjugate coeff
        if normalize
            return normal_order(new_term)
        else
            return dagger_term
        end
    end
end

function scale_term(term::Union{Term, Vector{Term}}, scale)::Union{Term, Vector{Term}}
    if isa(term, Vector{Term})
        return [scale_term(t, scale) for t in term]
    else
        new_term = deepcopy(term)
        new_term.coeff *= scale
        return new_term
    end
end

function Commutator_terms(A::Union{Term, Vector{Term}}, B::Union{Term, Vector{Term}}; scale=1)::Union{Term, Vector{Term}}
    terms_1 = multiply_terms(A, B)
    terms_2 = scale_term(multiply_terms(B, A), -1)
    terms::Vector{Term} = vcat(terms_1, terms_2)
    #terms = combine_terms(terms)
    
    if scale != 1
        return scale_term(terms, scale)
    else
        return terms
    end
end

function sigma_plus(index="")::Vector{Term}
    return [make_term("x"*index, 0.5), make_term("y"*index, 0.5im)]
end

function sigma_minus(index="")::Vector{Term}
    return [make_term("x"*index, 0.5), make_term("y"*index, -0.5im)]
end

sigma_x(index="") = make_term("x"*index)   # inline function definitions
sigma_y(index="") = make_term("y"*index)
sigma_z(index="") = make_term("z"*index)

function add_vars(term::Union{Term, Vector{Term}}, vars::Union{String, Vector{String}})
    if isa(vars, String)
        vars = String[vars]
    elseif !isa(vars, Vector{String})
        error("vars should be a string or a vector of strings")
    end

    if isa(term, Vector{Term})
        return [add_vars(t, vars) for t in term]
    else
        new_term = deepcopy(term)
        append!(new_term.vars, vars)
        return new_term
    end
end


# test
A = sigma_plus("i") 
B = dagger_term(sigma_minus("i"))
A = add_vars(A, "t")
B = add_vars(B, "t")
C = Commutator_terms(A, B)
C = scale_term(C, 0.5)
display(C)

4-element Vector{Term}:
 Term("yxz", "ijk", "", Int[], 0.5 + 0.0im, ["t", "t"])
 Term("xxz", "ijk", "", Int[], -0.0 - 0.5im, ["t", "t"])
 Term("yxz", "ijk", "", Int[], 0.5 - 0.0im, ["t", "t"])
 Term("xxz", "ijk", "", Int[], -0.0 - 0.5im, ["t", "t"])